In [27]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import pandas as pd
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split



def get_data():
    df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/AllCars.csv", skiprows=1)

    # Drop non-feature column
    df = df.drop(columns=["Make"])

    feature_names = ["Volume", "Doors"]
    class_column = "Style"

    X = df[feature_names]
    y = df[class_column]

    # Normalize features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

    # Train-test split
    features_train, features_test, classes_train, classes_test = train_test_split(
        X_scaled,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )

    class_names = sorted(y.unique())

    return (
        features_train,
        classes_train,
        features_test,
        classes_test,
        feature_names,
        class_names
    )
(
    features_train,
    classes_train,
    features_test,
    classes_test,
    feature_names,
    class_names
) = get_data()

features_train.head()
classes_train.head()
print(feature_names)
print(class_names)


['Volume', 'Doors']
['Jeep', 'Pickup', 'SUV', 'Sedan', 'Van']


In [29]:
features_train,classes_train,features_test,classes_test,feature_names,class_names = get_data()
feature_names,class_names,features_train,classes_train,features_test,classes_test


(['Volume', 'Doors'],
 ['Jeep', 'Pickup', 'SUV', 'Sedan', 'Van'],
        Volume     Doors
 15   0.255725  0.666667
 125  0.179389  0.000000
 11   0.351145  0.666667
 127  0.244275  0.666667
 51   0.259542  1.000000
 ..        ...       ...
 71   0.274809  0.666667
 106  0.274809  0.666667
 14   0.225191  0.666667
 92   0.278626  0.666667
 102  0.446565  0.666667
 
 [120 rows x 2 columns],
 15        SUV
 125     Sedan
 11     Pickup
 127     Sedan
 51        SUV
         ...  
 71      Sedan
 106     Sedan
 14      Sedan
 92        SUV
 102     Sedan
 Name: Style, Length: 120, dtype: object,
        Volume     Doors
 76   0.305344  0.666667
 18   0.293893  0.666667
 82   0.370229  1.000000
 81   0.358779  0.666667
 143  0.259542  0.666667
 31   0.293893  0.666667
 78   0.236641  0.666667
 64   0.221374  0.666667
 55   0.297710  0.666667
 85   0.374046  1.000000
 45   0.377863  0.666667
 12   0.274809  0.666667
 36   0.213740  0.666667
 9    0.221374  0.000000
 19   0.217557  0.666667


In [30]:
def get_data_from_csv(file_name):

    data = np.genfromtxt(file_name, delimiter=',', names=True, filling_values=0, dtype=None, \
ndmin=1)
    # print(f"Extracting from {file_name} and got {data}")
#----Extract all except the last column name as feature names
    feature_names = list(data.dtype.names[:-1])
    print(f"Feature names {feature_names}")
#----Extract unique values from last column as class names
    class_names = np.unique(data[data.dtype.names[-1]]).tolist()
    # print(f"Class names {class_names}")

#----Extract all except the last column as features
    features = data[feature_names].tolist()
    # print(f"Features are {features}")
#----Extract last column as classes
    np_classes = list(data[data.dtype.names[-1]])
    classes = [str(item) for item in np_classes]
    # print(f"All classes {classes}")

    return features,classes,feature_names,class_names

In [31]:
features_train,classes_train,feature_names,class_names = get_data_from_csv("/content/drive/MyDrive/Colab_Notebooks/training.csv")
features_test,classes_test,*_ = get_data_from_csv(file_name="/content/drive/MyDrive/Colab_Notebooks/testing.csv")
feature_names,class_names,features_train,classes_train,features_test,classes_test

Feature names ['Volume', 'Doors']
Feature names ['Volume', 'Doors']


(['Volume', 'Doors'],
 ['Jeep', 'Pickup', 'SUV', 'Sedan', 'Van'],
 [(0.2557251908396947, 0.6666666666666666),
  (0.17938931297709923, 0.0),
  (0.35114503816793896, 0.6666666666666666),
  (0.24427480916030533, 0.6666666666666666),
  (0.2595419847328244, 0.9999999999999999),
  (0.3740458015267175, 0.6666666666666666),
  (0.3129770992366412, 0.6666666666666666),
  (0.30534351145038163, 0.6666666666666666),
  (0.36641221374045796, 0.9999999999999999),
  (0.21755725190839695, 0.6666666666666666),
  (0.24045801526717556, 0.6666666666666666),
  (0.3244274809160306, 0.6666666666666666),
  (0.21755725190839695, 0.6666666666666666),
  (0.3702290076335878, 0.6666666666666666),
  (0.23664122137404578, 0.6666666666666666),
  (0.2748091603053435, 0.9999999999999999),
  (0.29389312977099236, 0.6666666666666666),
  (0.4885496183206106, 0.9999999999999999),
  (0.1679389312977099, 0.0),
  (0.2595419847328244, 0.6666666666666666),
  (0.32061068702290074, 0.6666666666666666),
  (0.21374045801526717, 0.666

In [32]:
def get_predictions(K,features_train,classes_train,features_test):

#----Initialize the K-NN Classifier
    knn = KNeighborsClassifier(n_neighbors=K)
#----Train the model
    knn.fit(features_train, classes_train)
#----Predict for the test data
    predictions = knn.predict(features_test)

    return knn,predictions

In [33]:
knn,predictions = get_predictions(4,features_train,classes_train,features_test)
knn,predictions

(KNeighborsClassifier(n_neighbors=4),
 array(['Sedan', 'Sedan', 'SUV', 'Pickup', 'SUV', 'Sedan', 'Sedan',
        'Sedan', 'Sedan', 'SUV', 'SUV', 'Sedan', 'Sedan', 'Sedan', 'Sedan',
        'Sedan', 'SUV', 'Pickup', 'Sedan', 'SUV', 'Sedan', 'Sedan', 'SUV',
        'SUV', 'Pickup', 'SUV', 'SUV', 'Pickup', 'SUV', 'Sedan', 'SUV'],
       dtype='<U6'))

In [34]:
def print_predictions(knn,predictions,features_test,classes_test):

#----Check the probability (How sure is the model?)
    probability = knn.predict_proba(features_test)
    confidences = np.max(probability, axis=1)
#----Output the result for each test data
    for index, value in enumerate(classes_test):
        print(f"The {classes_test[index]} is classified as: {predictions[index]}")
        print(f"Confidence is {confidences[index]}")

#----Compute the accuracy
    accuracy = accuracy_score(classes_test,predictions)
    print(f"The accuracy is {accuracy}")

In [35]:
print_predictions(knn,predictions,features_test,classes_test)

The Sedan is classified as: Sedan
Confidence is 0.5
The SUV is classified as: Sedan
Confidence is 1.0
The Jeep is classified as: SUV
Confidence is 1.0
The Pickup is classified as: Pickup
Confidence is 1.0
The Sedan is classified as: SUV
Confidence is 1.0
The SUV is classified as: Sedan
Confidence is 1.0
The Pickup is classified as: Sedan
Confidence is 1.0
The SUV is classified as: Sedan
Confidence is 0.75
The Sedan is classified as: Sedan
Confidence is 0.75
The SUV is classified as: SUV
Confidence is 1.0
The SUV is classified as: SUV
Confidence is 0.5
The Jeep is classified as: Sedan
Confidence is 0.75
The Sedan is classified as: Sedan
Confidence is 0.75
The Sedan is classified as: Sedan
Confidence is 0.75
The Sedan is classified as: Sedan
Confidence is 0.5
The Pickup is classified as: Sedan
Confidence is 0.75
The SUV is classified as: SUV
Confidence is 1.0
The SUV is classified as: Pickup
Confidence is 0.5
The Sedan is classified as: Sedan
Confidence is 1.0
The SUV is classified as: S

In [48]:
K_values = [1,2, 3,4, 5, 6 , 7,8, 9,10, 11]
accuracy_results = []

for K in K_values:
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(features_train, classes_train)

    predictions = knn.predict(features_test)
    accuracy = accuracy_score(classes_test, predictions)

    accuracy_results.append({
        "K": K,
        "Accuracy": accuracy
    })
accuracy_df = pd.DataFrame(accuracy_results)
accuracy_df.to_csv("Accuracy.csv", index=False)
from google.colab import files
files.download("Accuracy.csv")
accuracy_df




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,K,Accuracy
0,1,0.516129
1,2,0.483871
2,3,0.483871
3,4,0.612903
4,5,0.548387
5,6,0.580645
6,7,0.548387
7,8,0.580645
8,9,0.548387
9,10,0.548387


In [46]:
best_row = accuracy_df.loc[accuracy_df["Accuracy"].idxmax()]
best_K = int(best_row["K"])
best_accuracy = best_row["Accuracy"]

print("Best K:", best_K)
print("Best Accuracy:", best_accuracy)

Best K: 4
Best Accuracy: 0.6129032258064516


In [47]:
best_knn = KNeighborsClassifier(n_neighbors=best_K)
best_knn.fit(features_train, classes_train)


KNeighborsClassifier(n_neighbors=4)

In [50]:
best_row = accuracy_df.loc[accuracy_df["Accuracy"].idxmax()]
best_K = int(best_row["K"])

best_knn = KNeighborsClassifier(n_neighbors=best_K)
best_knn.fit(features_train, classes_train)

predictions = best_knn.predict(features_test)
probabilities = best_knn.predict_proba(features_test)
confidence = np.max(probabilities, axis=1)

testing_df = features_test.copy()
testing_df["Actual"] = classes_test.values
testing_df["Prediction"] = predictions
testing_df["Confidence"] = confidence

testing_df.to_csv("Testing.csv", index=False)
files.download("Testing.csv")
testing_df.head()



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Volume,Doors,Actual,Prediction,Confidence
76,0.305344,0.666667,Sedan,Sedan,0.5
18,0.293893,0.666667,SUV,Sedan,1.0
82,0.370229,1.000000,Jeep,SUV,1.0
81,0.358779,0.666667,Pickup,Pickup,1.0
143,0.259542,0.666667,Sedan,SUV,1.0
